In [2]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet

try: # disable certificate verification, needed on MacOS
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context
except ImportError:
    pass  # SSL module not available, skipping workaround

class SegmentationTransform:
    def __init__(self, size=(256, 256)):
        self.size = size
        self.image_transform = transforms.Compose([
            transforms.Resize(self.size),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize(self.size, interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor()
        ])

    def __call__(self, image, target):
        image = self.image_transform(image)
        target = self.mask_transform(target).squeeze(0).long() - 1  # [1, H, W] → [H, W], label remap
        return image, target

train_dataset = OxfordIIITPet(
    root='.',
    target_types='segmentation',
    download=True,
    transforms=SegmentationTransform(size=(256, 256))
)

batch_size = 32  # TODO: change to your needs

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

def load_images(filename: str, image_shape: tuple = (3, 256, 256)):
    df = pd.read_csv(filename)

    images = []
    for _, row in df.iterrows():
        flat = eval(row['image'])  # Convert stringified list back to list
        tensor = torch.tensor(flat, dtype=torch.float32).reshape(image_shape)
        images.append(tensor)

    return torch.stack(images)  # [B, C, H, W]



ModuleNotFoundError: No module named 'torch'